# 04 Forecasting models

This notebook compares a few simple forecasting baselines with machine learning models for short-term activation planning. I kept the simple models because they are useful benchmarks; a more complex model only helps if it actually improves the test results.

The forecasts here should be read as planning estimates, not exact predictions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# tree models are included as a practical benchmark, not because they are automatically better
# keeping imports together so the notebook is easier to rerun
# 1) Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
# 2) Optional: install Prophet if needed

try:
    from prophet import Prophet
    print("Prophet is already installed.")
except:
    print("Prophet is not installed. Installing now...")
    !pip install prophet -q
    from prophet import Prophet
    print("Prophet installed and imported.")

In [ ]:
# 3) Set file paths

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Practicum/Demo File")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
VISUAL_DIR = BASE_DIR / "visuals"

VISUAL_DIR.mkdir(parents=True, exist_ok=True)

clean_daily_path = PROCESSED_DIR / "clean_daily_master.csv"
clean_monthly_path = PROCESSED_DIR / "clean_monthly_kpi.csv"

print("Clean daily file exists:", clean_daily_path.exists())
print("Clean monthly file exists:", clean_monthly_path.exists())

In [ ]:
# 4) Load clean data

daily = pd.read_csv(clean_daily_path)
monthly = pd.read_csv(clean_monthly_path)

daily["Date"] = pd.to_datetime(daily["Date"])
monthly["Month"] = pd.to_datetime(monthly["Month"])

print("Daily shape:", daily.shape)
print("Monthly shape:", monthly.shape)

display(daily.head())
display(monthly.head())

In [ ]:
# 5) Create market-level daily and monthly activation tables

# I am forecasting the full market first because market-level data is usually
# more stable than individual store-level daily data.

daily_market = (
    daily
    .groupby("Date", as_index=False)
    .agg(Total_Activation=("Total_Activation", "sum"))
    .sort_values("Date")
)


daily_market = daily_market.rename(columns={
    "Date": "ds",
    "Total_Activation": "y"
})

monthly_market = daily_market.copy()
monthly_market["Month"] = monthly_market["ds"].dt.to_period("M").dt.to_timestamp()

monthly_market = (
    monthly_market
    .groupby("Month", as_index=False)
    .agg(y=("y", "sum"))
    .rename(columns={"Month": "ds"})
)

print("Daily market shape:", daily_market.shape)
print("Monthly market shape:", monthly_market.shape)

display(daily_market.head())
display(monthly_market.head())

In [ ]:
# 6) Plot historical monthly trend

plt.figure(figsize=(14, 6))
plt.plot(monthly_market["ds"], monthly_market["y"], marker="o")

plt.title("Monthly Total Activation Trend")
plt.xlabel("Month")
plt.ylabel("Total Activation")
plt.grid(True)
plt.tight_layout()


plt.savefig(VISUAL_DIR / "forecast_monthly_activation_trend.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 7) Plot historical daily trend

plt.figure(figsize=(14, 6))
plt.plot(daily_market["ds"], daily_market["y"])

plt.title("Daily Total Activation Trend")
plt.xlabel("Date")
plt.ylabel("Total Activation")
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "forecast_daily_activation_trend.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 8) Forecast evaluation function

def evaluate_forecast(actual, predicted, model_name):
    """
    Calculates common forecast error metrics.
    Lower MAE, RMSE, and MAPE are better.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    # Avoid divide-by-zero issue
    if np.any(actual == 0):
        mape = np.nan
    else:
        mape = np.mean(np.abs((actual - predicted) / actual)) * 100

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

In [ ]:
# 9) Monthly train/test split

# Monthly data is smoother and useful for planning.
# I use the last 6 months as the test period.

monthly_test_months = 6

monthly_train = monthly_market.iloc[:-monthly_test_months].copy()
monthly_test = monthly_market.iloc[-monthly_test_months:].copy()

print("Monthly train range:", monthly_train["ds"].min(), "to", monthly_train["ds"].max())
print("Monthly test range:", monthly_test["ds"].min(), "to", monthly_test["ds"].max())

display(monthly_train.tail())
display(monthly_test)

In [ ]:
# 10) Monthly baseline models

monthly_pred = monthly_test.copy()

# Naive forecast: use last observed training month
monthly_pred["Naive_Forecast"] = monthly_train["y"].iloc[-1]

# Moving averages
monthly_pred["Moving_Avg_3"] = monthly_train["y"].tail(3).mean()
monthly_pred["Moving_Avg_6"] = monthly_train["y"].tail(6).mean()

# Linear trend
train_index = np.arange(len(monthly_train))
test_index = np.arange(len(monthly_train), len(monthly_train) + len(monthly_test))

slope, intercept = np.polyfit(train_index, monthly_train["y"], 1)
monthly_pred["Linear_Trend"] = slope * test_index + intercept

display(monthly_pred)

In [ ]:
# 11) Monthly exponential smoothing

# This is still a simple time-series model, but a little more flexible
# than a flat moving average.

from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing

monthly_train_ts = monthly_train.set_index("ds")["y"]

try:
    ses_model = SimpleExpSmoothing(
        monthly_train_ts,
        initialization_method="estimated"
    ).fit(optimized=True)

    monthly_pred["Simple_Exp_Smoothing"] = ses_model.forecast(len(monthly_test)).values

except Exception as e:
    print("SES model had an issue:", e)
    monthly_pred["Simple_Exp_Smoothing"] = np.nan


try:
    holt_model = ExponentialSmoothing(
        monthly_train_ts,
        trend="add",
        seasonal=None,
        initialization_method="estimated"
    ).fit(optimized=True)

    monthly_pred["Holt_Trend"] = holt_model.forecast(len(monthly_test)).values


except Exception as e:
    print("Holt model had an issue:", e)
    monthly_pred["Holt_Trend"] = np.nan


display(monthly_pred)


In [ ]:
# 12) Evaluate monthly models

monthly_model_cols = [
    ("Naive Forecast", "Naive_Forecast"),
    ("3-Month Moving Average", "Moving_Avg_3"),
    ("6-Month Moving Average", "Moving_Avg_6"),
    ("Linear Trend", "Linear_Trend"),
    ("Simple Exponential Smoothing", "Simple_Exp_Smoothing"),
    ("Holt Trend", "Holt_Trend")
]

monthly_results = []

for model_name, col in monthly_model_cols:
    temp = monthly_pred.dropna(subset=[col]).copy()
    result = evaluate_forecast(temp["y"], temp[col], model_name)
    result["Data_Level"] = "Monthly"
    monthly_results.append(result)

monthly_results_df = pd.DataFrame(monthly_results).sort_values("MAE").reset_index(drop=True)

display(monthly_results_df)

In [ ]:
# 13) Plot monthly actual vs best model

monthly_best_model = monthly_results_df.iloc[0]["Model"]

monthly_model_map = {
    "Naive Forecast": "Naive_Forecast",
    "3-Month Moving Average": "Moving_Avg_3",
    "6-Month Moving Average": "Moving_Avg_6",
    "Linear Trend": "Linear_Trend",
    "Simple Exponential Smoothing": "Simple_Exp_Smoothing",
    "Holt Trend": "Holt_Trend"
}

monthly_best_col = monthly_model_map[monthly_best_model]

print("Best monthly model:", monthly_best_model)

plt.figure(figsize=(12, 5))
plt.plot(monthly_pred["ds"], monthly_pred["y"], marker="o", label="Actual")
plt.plot(monthly_pred["ds"], monthly_pred[monthly_best_col], marker="o", linestyle="--", label=monthly_best_model)

plt.title(f"Monthly Forecast: Actual vs {monthly_best_model}")
plt.xlabel("Month")
plt.ylabel("Total Activation")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "monthly_forecast_actual_vs_best_model.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
''' DAILY FORECASTING WITH BASELINE MDOELS'''

# 14) Daily train/test split

# Daily data has many more observations, so I can test machine learning models.
# I use the last 90 days as the test period.

daily_test_days = 90

daily_train = daily_market.iloc[:-daily_test_days].copy()
daily_test = daily_market.iloc[-daily_test_days:].copy()

print("Daily train range:", daily_train["ds"].min(), "to", daily_train["ds"].max())
print("Daily test range:", daily_test["ds"].min(), "to", daily_test["ds"].max())

display(daily_train.tail())
display(daily_test.head())

In [ ]:
# 15) Daily baseline models

daily_pred = daily_test.copy()

daily_pred["Naive_Forecast"] = daily_train["y"].iloc[-1]
daily_pred["Moving_Avg_7"] = daily_train["y"].tail(7).mean()
daily_pred["Moving_Avg_30"] = daily_train["y"].tail(30).mean()

# Seasonal naive:
# For each date, use the value from the same weekday one week earlier.
full_daily = daily_market.copy()
full_daily["Seasonal_Naive_7"] = full_daily["y"].shift(7)

daily_pred = daily_pred.merge(
    full_daily[["ds", "Seasonal_Naive_7"]],
    on="ds",
    how="left"
)

display(daily_pred.head())

In [ ]:
# 16) Evaluate daily baseline models

daily_baseline_results = []

daily_baseline_cols = [
    ("Naive Forecast", "Naive_Forecast"),
    ("7-Day Moving Average", "Moving_Avg_7"),
    ("30-Day Moving Average", "Moving_Avg_30"),
    ("Seasonal Naive 7-Day", "Seasonal_Naive_7")
]

for model_name, col in daily_baseline_cols:
    temp = daily_pred.dropna(subset=[col]).copy()
    result = evaluate_forecast(temp["y"], temp[col], model_name)
    result["Data_Level"] = "Daily"
    daily_baseline_results.append(result)

daily_baseline_results_df = pd.DataFrame(daily_baseline_results)

display(daily_baseline_results_df.sort_values("MAE"))

In [ ]:
# 17) Create daily lag and calendar features

# Lag features use past activation values to predict current activation.
# Example: lag_7 means activation from the same weekday last week.

ml_df = daily_market.copy()

ml_df["day_of_week"] = ml_df["ds"].dt.dayofweek
ml_df["day_of_month"] = ml_df["ds"].dt.day
ml_df["month"] = ml_df["ds"].dt.month
ml_df["quarter"] = ml_df["ds"].dt.quarter
ml_df["is_weekend"] = ml_df["day_of_week"].isin([5, 6]).astype(int)

# Lag features
ml_df["lag_1"] = ml_df["y"].shift(1)
ml_df["lag_2"] = ml_df["y"].shift(2)
ml_df["lag_3"] = ml_df["y"].shift(3)
ml_df["lag_7"] = ml_df["y"].shift(7)
ml_df["lag_14"] = ml_df["y"].shift(14)
ml_df["lag_30"] = ml_df["y"].shift(30)

# Rolling features. I use shift(1) so today's value is not used to predict today.
ml_df["rolling_7_mean"] = ml_df["y"].shift(1).rolling(window=7).mean()
ml_df["rolling_14_mean"] = ml_df["y"].shift(1).rolling(window=14).mean()
ml_df["rolling_30_mean"] = ml_df["y"].shift(1).rolling(window=30).mean()

ml_df["rolling_7_std"] = ml_df["y"].shift(1).rolling(window=7).std()
ml_df["rolling_30_std"] = ml_df["y"].shift(1).rolling(window=30).std()

ml_df = ml_df.dropna().reset_index(drop=True)

print("ML daily dataset shape:", ml_df.shape)
display(ml_df.head())

In [ ]:
# 18) Create ML train/test split

ml_train = ml_df[ml_df["ds"] < daily_test["ds"].min()].copy()
ml_test = ml_df[ml_df["ds"] >= daily_test["ds"].min()].copy()

feature_cols = [
    "day_of_week",
    "day_of_month",
    "month",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_7_mean",
    "rolling_14_mean",
    "rolling_30_mean",
    "rolling_7_std",
    "rolling_30_std"
]

X_train = ml_train[feature_cols]
y_train = ml_train["y"]

X_test = ml_test[feature_cols]
y_test = ml_test["y"]

print("ML train shape:", X_train.shape)
print("ML test shape:", X_test.shape)

In [ ]:
# 18) Create ML train/test split

ml_train = ml_df[ml_df["ds"] < daily_test["ds"].min()].copy()
ml_test = ml_df[ml_df["ds"] >= daily_test["ds"].min()].copy()

feature_cols = [
    "day_of_week",
    "day_of_month",
    "month",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_7_mean",
    "rolling_14_mean",
    "rolling_30_mean",
    "rolling_7_std",
    "rolling_30_std"
]

X_train = ml_train[feature_cols]
y_train = ml_train["y"]

X_test = ml_test[feature_cols]
y_test = ml_test["y"]

print("ML train shape:", X_train.shape)
print("ML test shape:", X_test.shape)

In [ ]:
# 19) Random Forest model

# Random Forest is useful here because it can capture nonlinear patterns
# from lag features and calendar variables without much manual tuning.

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

ml_test["Random_Forest_Forecast"] = rf_model.predict(X_test)

rf_result = evaluate_forecast(
    ml_test["y"],
    ml_test["Random_Forest_Forecast"],
    "Random Forest"
)

rf_result["Data_Level"] = "Daily"

rf_result

In [ ]:
# XGBoost is useful here because the daily data has lag and calendar patterns
# 20) XGBoost model

# XGBoost is another strong model for tabular forecasting features.
# I keep the parameters fairly simple because this is a demo project.

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_model.fit(X_train, y_train)

ml_test["XGBoost_Forecast"] = xgb_model.predict(X_test)

xgb_result = evaluate_forecast(
    ml_test["y"],
    ml_test["XGBoost_Forecast"],
    "XGBoost"
)

xgb_result["Data_Level"] = "Daily"

xgb_result

In [ ]:
# 21) Prophet model

# Prophet is useful for daily forecasting because it can model trend
# and weekly/yearly seasonality. Here I use only the daily market series.

prophet_train = daily_train.copy()
prophet_test = daily_test.copy()

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)

prophet_model.fit(prophet_train)

future_test = prophet_model.make_future_dataframe(periods=daily_test_days, freq="D")

prophet_forecast = prophet_model.predict(future_test)

prophet_test_pred = prophet_forecast[
    prophet_forecast["ds"].isin(prophet_test["ds"])
][["ds", "yhat"]].copy()

prophet_test_pred = prophet_test_pred.rename(columns={"yhat": "Prophet_Forecast"})

daily_pred = daily_pred.merge(prophet_test_pred, on="ds", how="left")

prophet_result = evaluate_forecast(
    daily_pred["y"],
    daily_pred["Prophet_Forecast"],
    "Prophet"
)

prophet_result["Data_Level"] = "Daily"

prophet_result

In [ ]:
# 22) Plot Prophet components

# This plot is useful for checking whether Prophet sees weekly/yearly patterns.
# It may create multiple charts automatically.

prophet_model.plot_components(prophet_forecast)
plt.tight_layout()
plt.savefig(VISUAL_DIR / "prophet_components.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 23) Combine all daily model results

daily_model_results = pd.concat(
    [
        daily_baseline_results_df,
        pd.DataFrame([rf_result, xgb_result, prophet_result])
    ],
    ignore_index=True
)

daily_model_results = daily_model_results.sort_values("MAE").reset_index(drop=True)

display(daily_model_results)

In [ ]:
# 24) Combine daily predictions

daily_comparison = daily_pred[
    [
        "ds",
        "y",
        "Naive_Forecast",
        "Moving_Avg_7",
        "Moving_Avg_30",
        "Seasonal_Naive_7",
        "Prophet_Forecast"
    ]
].copy()

daily_comparison = daily_comparison.merge(
    ml_test[["ds", "Random_Forest_Forecast", "XGBoost_Forecast"]],
    on="ds",
    how="left"
)

display(daily_comparison.head())

In [ ]:
# 25) Plot actual vs selected daily models

plt.figure(figsize=(14, 6))

plt.plot(daily_comparison["ds"], daily_comparison["y"], label="Actual", linewidth=2)
plt.plot(daily_comparison["ds"], daily_comparison["Seasonal_Naive_7"], label="Seasonal Naive 7-Day", linestyle="--")
plt.plot(daily_comparison["ds"], daily_comparison["Random_Forest_Forecast"], label="Random Forest", linestyle="--")
plt.plot(daily_comparison["ds"], daily_comparison["XGBoost_Forecast"], label="XGBoost", linestyle="--")
plt.plot(daily_comparison["ds"], daily_comparison["Prophet_Forecast"], label="Prophet", linestyle="--")

plt.title("Daily Forecast Model Comparison")
plt.xlabel("Date")
plt.ylabel("Total Activation")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "daily_forecast_model_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 26) Plot actual vs best daily model

daily_best_model = daily_model_results.iloc[0]["Model"]

daily_model_map = {
    "Naive Forecast": "Naive_Forecast",
    "7-Day Moving Average": "Moving_Avg_7",
    "30-Day Moving Average": "Moving_Avg_30",
    "Seasonal Naive 7-Day": "Seasonal_Naive_7",
    "Random Forest": "Random_Forest_Forecast",
    "XGBoost": "XGBoost_Forecast",
    "Prophet": "Prophet_Forecast"
}

daily_best_col = daily_model_map[daily_best_model]

print("Best daily model:", daily_best_model)

plt.figure(figsize=(14, 6))
plt.plot(daily_comparison["ds"], daily_comparison["y"], label="Actual", linewidth=2)
plt.plot(daily_comparison["ds"], daily_comparison[daily_best_col], label=f"Forecast: {daily_best_model}", linestyle="--")

plt.title(f"Daily Forecast: Actual vs {daily_best_model}")
plt.xlabel("Date")
plt.ylabel("Total Activation")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "daily_forecast_actual_vs_best_model.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
''' FEATURE IMPORTANCE'''
# 27) Random Forest feature importance

rf_feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(rf_feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(rf_feature_importance["Feature"], rf_feature_importance["Importance"])
plt.gca().invert_yaxis()

plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "rf_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 28) XGBoost feature importance

xgb_feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": xgb_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(xgb_feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(xgb_feature_importance["Feature"], xgb_feature_importance["Importance"])
plt.gca().invert_yaxis()

plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "xgb_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
''' 30-DAY FORECAST'''
# 29) Helper function for recursive ML forecasting

def create_features_for_future_date(history_df, forecast_date):
    """
    Creates one row of future features using the history available so far.
    For recursive forecasting, predicted values get added back to history.
    """
    temp = history_df.sort_values("ds").copy()

    row = {
        "ds": forecast_date,
        "day_of_week": forecast_date.dayofweek,
        "day_of_month": forecast_date.day,
        "month": forecast_date.month,
        "quarter": forecast_date.quarter,
        "is_weekend": int(forecast_date.dayofweek in [5, 6]),
        "lag_1": temp["y"].iloc[-1],
        "lag_2": temp["y"].iloc[-2],
        "lag_3": temp["y"].iloc[-3],
        "lag_7": temp["y"].iloc[-7],
        "lag_14": temp["y"].iloc[-14],
        "lag_30": temp["y"].iloc[-30],
        "rolling_7_mean": temp["y"].tail(7).mean(),
        "rolling_14_mean": temp["y"].tail(14).mean(),
        "rolling_30_mean": temp["y"].tail(30).mean(),
        "rolling_7_std": temp["y"].tail(7).std(),
        "rolling_30_std": temp["y"].tail(30).std()
    }

    return pd.DataFrame([row])

In [ ]:
# 30) Generate 30-day future forecast

future_days = 30
last_date = daily_market["ds"].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=future_days, freq="D")

# Refit / use the best daily model logic.
# If best is RF or XGB, use recursive forecasting.
# If best is Prophet, use Prophet's future dataframe.

if daily_best_model == "Random Forest":
    final_model = rf_model
    full_history = daily_market.copy().sort_values("ds").reset_index(drop=True)
    future_rows = []

    for future_date in future_dates:
        next_features = create_features_for_future_date(full_history, future_date)
        next_pred = final_model.predict(next_features[feature_cols])[0]
        next_pred = max(0, next_pred)

        future_rows.append({
            "ds": future_date,
            "Forecast": next_pred,
            "Selected_Model": daily_best_model
        })

        full_history = pd.concat(
            [full_history, pd.DataFrame({"ds": [future_date], "y": [next_pred]})],
            ignore_index=True
        )

    future_daily_forecast = pd.DataFrame(future_rows)

elif daily_best_model == "XGBoost":
    final_model = xgb_model
    full_history = daily_market.copy().sort_values("ds").reset_index(drop=True)
    future_rows = []

    for future_date in future_dates:
        next_features = create_features_for_future_date(full_history, future_date)
        next_pred = final_model.predict(next_features[feature_cols])[0]
        next_pred = max(0, next_pred)

        future_rows.append({
            "ds": future_date,
            "Forecast": next_pred,
            "Selected_Model": daily_best_model
        })

        full_history = pd.concat(
            [full_history, pd.DataFrame({"ds": [future_date], "y": [next_pred]})],
            ignore_index=True
        )

    future_daily_forecast = pd.DataFrame(future_rows)

elif daily_best_model == "Prophet":
    prophet_future = prophet_model.make_future_dataframe(periods=future_days, freq="D")
    prophet_future_forecast = prophet_model.predict(prophet_future)

    future_daily_forecast = prophet_future_forecast.tail(future_days)[["ds", "yhat"]].copy()
    future_daily_forecast = future_daily_forecast.rename(columns={"yhat": "Forecast"})
    future_daily_forecast["Selected_Model"] = daily_best_model

else:
    # Fallback for baseline models
    if daily_best_model == "Naive Forecast":
        forecast_values = [daily_market["y"].iloc[-1]] * future_days

    elif daily_best_model == "7-Day Moving Average":
        forecast_values = [daily_market["y"].tail(7).mean()] * future_days

    elif daily_best_model == "30-Day Moving Average":
        forecast_values = [daily_market["y"].tail(30).mean()] * future_days

    elif daily_best_model == "Seasonal Naive 7-Day":
        recent_week = daily_market["y"].tail(7).values
        forecast_values = [recent_week[i % 7] for i in range(future_days)]

    future_daily_forecast = pd.DataFrame({
        "ds": future_dates,
        "Forecast": forecast_values,
        "Selected_Model": daily_best_model
    })

display(future_daily_forecast.head())
display(future_daily_forecast.tail())

In [ ]:
# 31) Plot future 30-day forecast

recent_history = daily_market.tail(120).copy()

plt.figure(figsize=(14, 6))
plt.plot(recent_history["ds"], recent_history["y"], label="Recent Actual", linewidth=2)
plt.plot(future_daily_forecast["ds"], future_daily_forecast["Forecast"], label="30-Day Forecast", linestyle="--", marker="o")

plt.title(f"30-Day Daily Activation Forecast Using {daily_best_model}")
plt.xlabel("Date")
plt.ylabel("Total Activation")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "future_30_day_daily_forecast.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 32) Save forecasting outputs

monthly_results_path = PROCESSED_DIR / "monthly_forecasting_model_results.csv"
monthly_predictions_path = PROCESSED_DIR / "monthly_forecast_test_predictions.csv"

daily_results_path = PROCESSED_DIR / "daily_forecasting_model_results.csv"
daily_predictions_path = PROCESSED_DIR / "daily_forecast_test_predictions.csv"
future_daily_path = PROCESSED_DIR / "daily_30_day_future_forecast.csv"

rf_importance_path = PROCESSED_DIR / "rf_feature_importance.csv"
xgb_importance_path = PROCESSED_DIR / "xgb_feature_importance.csv"

monthly_results_df.to_csv(monthly_results_path, index=False)
monthly_pred.to_csv(monthly_predictions_path, index=False)

daily_model_results.to_csv(daily_results_path, index=False)
daily_comparison.to_csv(daily_predictions_path, index=False)
future_daily_forecast.to_csv(future_daily_path, index=False)

rf_feature_importance.to_csv(rf_importance_path, index=False)
xgb_feature_importance.to_csv(xgb_importance_path, index=False)


print("Saved monthly model results:", monthly_results_path)
print("Saved daily model results:", daily_results_path)
print("Saved future daily forecast:", future_daily_path)

In [ ]:
# 32) Save forecasting outputs

monthly_results_path = PROCESSED_DIR / "monthly_forecasting_model_results.csv"
monthly_predictions_path = PROCESSED_DIR / "monthly_forecast_test_predictions.csv"

daily_results_path = PROCESSED_DIR / "daily_forecasting_model_results.csv"
daily_predictions_path = PROCESSED_DIR / "daily_forecast_test_predictions.csv"
future_daily_path = PROCESSED_DIR / "daily_30_day_future_forecast.csv"

rf_importance_path = PROCESSED_DIR / "rf_feature_importance.csv"
xgb_importance_path = PROCESSED_DIR / "xgb_feature_importance.csv"

monthly_results_df.to_csv(monthly_results_path, index=False)
monthly_pred.to_csv(monthly_predictions_path, index=False)

daily_model_results.to_csv(daily_results_path, index=False)
daily_comparison.to_csv(daily_predictions_path, index=False)
future_daily_forecast.to_csv(future_daily_path, index=False)

rf_feature_importance.to_csv(rf_importance_path, index=False)
xgb_feature_importance.to_csv(xgb_importance_path, index=False)

print("Saved monthly model results:", monthly_results_path)
print("Saved daily model results:", daily_results_path)
print("Saved future daily forecast:", future_daily_path)